## Bag of Words and TF-IDF

This notebook implements and evaluates two classical text representation techniques for sentiment classification:

1. Bag of Words (BoW)
2. TF-IDF

The dataset has already been divided into:

- **72% Train** → used to fit vectorizers and classifiers
- **8% Validation** → used for model selection and hyperparameter tuning
- **20% Test** → reserved for final evaluation

The validation set is used to make modelling decisions, while the test set is used only after the final configuration has been selected.

In [57]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

### Loading data

In [58]:
train_df = pd.read_csv("D:/NLP-Projects/sentiment-analysis-imdb/data/processed/train.csv")

validation_df = pd.read_csv("D:/NLP-Projects/sentiment-analysis-imdb/data/processed/validation.csv")

test_df = pd.read_csv("D:/NLP-Projects/sentiment-analysis-imdb/data/processed/test.csv")

In [59]:
print("Train shape:      ", train_df.shape)
print("Validation shape: ", validation_df.shape)
print("Test shape:       ", test_df.shape)

Train shape:       (35698, 7)
Validation shape:  (3967, 7)
Test shape:        (9917, 7)


### Seperating features and labels

In [60]:
# BoW / TF-IDF
X_train_clean = train_df["clean_review"]
X_val_clean = validation_df["clean_review"]
X_test_clean = test_df["clean_review"]

# Labels
y_train = train_df["sentiment"]
y_val = validation_df["sentiment"]
y_test = test_df["sentiment"]

In [61]:

def evaluate_model(y_test,y_pred, case):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"{case} Accuracy : {accuracy:.4f}")
    print(f"{case} Precision: {precision:.4f}")
    print(f"{case} Recall   : {recall:.4f}")
    print(f"{case} F1 Score : {f1:.4f}")

## 1. Bag of Words (BoW)

Bag of Words represents a document using the frequency of words appearing in the vocabulary.

For example:

Document 1:
"good movie good"

Document 2:
"bad movie"

Vocabulary:

["good", "movie", "bad"]

The representation becomes:

| Document | good | movie | bad |
|----------|------|-------|-----|
| Doc 1 | 2 | 1 | 0 |
| Doc 2 | 0 | 1 | 1 |

BoW ignores the order of words and focuses on whether and how frequently words occur.

The resulting matrix is usually **high-dimensional and sparse**.

### Creating and fitting the BoW vectorizer

In [62]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

bow_pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000, random_state =42))
])

bow_pipeline.fit(X_train_clean, y_train)

y_val_bow_pred = bow_pipeline.predict(X_val_clean)


In [63]:
print("Vocabulary size:", len(bow_pipeline.named_steps["vectorizer"].vocabulary_))

Vocabulary size: 82824


In [64]:
print("\nFirst 20 vocabulary terms:")
print(list(bow_pipeline.named_steps["vectorizer"].vocabulary_.keys())[:20])


First 20 vocabulary terms:
['geesh', 'never', 'ever', 'thought', 'would', 'write', 'four', 'word', 'actually', 'highpoint', 'little', 'flick', 'movie', 'packaged', 'rented', 'supposedly', 'comedy', 'girl', 'kidnapped', 'not']


### Validation evaluation

In [65]:
evaluate_model(y_val, y_val_bow_pred, "validation")

validation Accuracy : 0.8755
validation Precision: 0.8711
validation Recall   : 0.8825
validation F1 Score : 0.8767


In [66]:
print(classification_report(y_val, y_val_bow_pred))

              precision    recall  f1-score   support

           0       0.88      0.87      0.87      1976
           1       0.87      0.88      0.88      1991

    accuracy                           0.88      3967
   macro avg       0.88      0.88      0.88      3967
weighted avg       0.88      0.88      0.88      3967



## BoW Hyperparameter Tuning

The validation set allows us to compare different BoW configurations without touching the test set.

Possible BoW hyperparameters include:

- `ngram_range`
- `min_df`
- `max_df`
- `max_features`

We fit each candidate vectorizer only on the training data, transform the validation data, train the classifier on the training representations, and compare validation performance.

The test set remains untouched during this process.

In [67]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd


bow_configs = {
    "unigram": {
        "vectorizer__ngram_range": (1, 1),
    },

    "unigram_bigram": {
        "vectorizer__ngram_range": (1, 2),
    },

    "unigram_bigram_30000": {
        "vectorizer__ngram_range": (1, 2),
        "vectorizer__max_features": 30000,
    },

    "unigram_bigram_min_df2": {
        "vectorizer__ngram_range": (1, 2),
        "vectorizer__min_df": 2,
    },
}

In [68]:
bow_results = []

for name, params in bow_configs.items():

    bow_pipeline = Pipeline([
        ("vectorizer", CountVectorizer()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    # Apply the current configuration
    bow_pipeline.set_params(**params)

    # Fit only on training data
    bow_pipeline.fit(X_train_clean, y_train)

    # Evaluate on validation data
    y_val_pred = bow_pipeline.predict(X_val_clean)

    # Access fitted vectorizer inside pipeline
    vocabulary_size = len(
        bow_pipeline.named_steps["vectorizer"].vocabulary_
    )

    bow_results.append({
        "configuration": name,
        "vocabulary_size": vocabulary_size,
        "accuracy": accuracy_score(y_val, y_val_pred),
        "precision": precision_score(y_val, y_val_pred),
        "recall": recall_score(y_val, y_val_pred),
        "f1": f1_score(y_val, y_val_pred)
    })


bow_results_df = pd.DataFrame(bow_results)

bow_results_df.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

,configuration,vocabulary_size,accuracy,precision,recall,f1
0,unigram_bigram_min_df2,482094,0.894379,0.890268,0.900552,0.895381
1,unigram_bigram,2322536,0.892362,0.884464,0.903566,0.893913
2,unigram_bigram_30000,30000,0.890345,0.890171,0.891512,0.890841
3,unigram,82824,0.875473,0.871096,0.882471,0.876747


### BoW Observations

- The unigram + bigram representation performed better than the unigram-only representation, indicating that combinations of two consecutive words provided useful information for sentiment classification.
- Applying `min_df=2` improved the validation performance of the unigram + bigram BoW representation while substantially reducing the vocabulary size. The F1 score increased from approximately **0.8939 to 0.8954**, while the vocabulary decreased from approximately **2.32 million to 482 thousand features**.
- Limiting the vocabulary to `max_features=30,000` reduced the feature space considerably, but resulted in a slightly lower validation F1 score than the `min_df=2` configuration.
- Among the BoW configurations tested, **unigram + bigram with `min_df=2`** achieved the best validation F1 score.
---

# 2. TF-IDF

TF-IDF stands for **Term Frequency–Inverse Document Frequency**.

Unlike BoW, which mainly represents how frequently a word appears, TF-IDF also considers how common the word is across the entire corpus.

A word receives a higher weight when:

- it occurs frequently in a particular document
- but does not occur in too many other documents

Very common words receive lower weights because they provide less information for distinguishing documents.

The representation is still sparse, but the values are now weighted rather than simple word counts.

### Creating and fitting Tfidf Vectorizer

In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

tfidf_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

tfidf_pipeline.fit(X_train_clean, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizer', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [70]:
y_val_tfidf_pred = tfidf_pipeline.predict(X_val_clean)

In [71]:
print(
    "Vocabulary size:",
    len(tfidf_pipeline.named_steps["vectorizer"].vocabulary_)
)

Vocabulary size: 82824


### Validation evaluation

In [72]:
tfidf_val_accuracy = accuracy_score(y_val, y_val_tfidf_pred)
tfidf_val_precision = precision_score(y_val, y_val_tfidf_pred)
tfidf_val_recall = recall_score(y_val, y_val_tfidf_pred)
tfidf_val_f1 = f1_score(y_val, y_val_tfidf_pred)

print(f"Validation Accuracy : {tfidf_val_accuracy:.4f}")
print(f"Validation Precision: {tfidf_val_precision:.4f}")
print(f"Validation Recall   : {tfidf_val_recall:.4f}")
print(f"Validation F1 Score : {tfidf_val_f1:.4f}")

Validation Accuracy : 0.8833
Validation Precision: 0.8716
Validation Recall   : 0.9001
Validation F1 Score : 0.8856


## TF-IDF Hyperparameter Tuning

We can now use the validation set to compare different TF-IDF configurations.

Important hyperparameters include:

- `ngram_range`
- `min_df`
- `max_df`
- `max_features`
- `sublinear_tf`

For every configuration:

1. Fit TF-IDF on the training data.
2. Transform the training data.
3. Transform the validation data.
4. Train the classifier using only training data.
5. Evaluate on validation data.
6. Compare the results.

The test set remains untouched.

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf_configs = {
    "unigram": {
        "vectorizer__ngram_range": (1, 1),
    },

    "unigram_bigram": {
        "vectorizer__ngram_range": (1, 2),
    },

    "unigram_bigram_min_df2": {
        "vectorizer__ngram_range": (1, 2),
        "vectorizer__min_df": 2,
    },

    "unigram_bigram_sublinear": {
        "vectorizer__ngram_range": (1, 2),
        "vectorizer__sublinear_tf": True,
    },
}


In [74]:
tfidf_results = []

for name, params in tfidf_configs.items():

    tfidf_pipeline = Pipeline([
        ("vectorizer", TfidfVectorizer()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    tfidf_pipeline.set_params(**params)

    tfidf_pipeline.fit(X_train_clean, y_train)

    y_val_pred = tfidf_pipeline.predict(X_val_clean)

    vocabulary_size = len(
        tfidf_pipeline.named_steps["vectorizer"].vocabulary_
    )

    tfidf_results.append({
        "configuration": name,
        "vocabulary_size": vocabulary_size,
        "accuracy": accuracy_score(y_val, y_val_pred),
        "precision": precision_score(y_val, y_val_pred),
        "recall": recall_score(y_val, y_val_pred),
        "f1": f1_score(y_val, y_val_pred)
    })


tfidf_results_df = pd.DataFrame(tfidf_results)

tfidf_results_df.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

,configuration,vocabulary_size,accuracy,precision,recall,f1
0,unigram_bigram_min_df2,482094,0.885304,0.872093,0.904068,0.887793
1,unigram,82824,0.883287,0.871595,0.900050,0.885594
2,unigram_bigram_sublinear,2322536,0.879506,0.867411,0.897037,0.881975
3,unigram_bigram,2322536,0.878498,0.864669,0.898543,0.881281


### TF-IDF Observations

- Adding bigrams did not improve TF-IDF performance compared with the unigram configuration in the tested settings.
- Applying `min_df=2` to the unigram + bigram representation produced the best TF-IDF validation performance among the tested configurations.
- `sublinear_tf=True` did not improve the validation performance in this experiment.
- The best TF-IDF configuration achieved a validation F1 score of approximately **0.888**.
----

### Comparing BoW and TF-IDF

In [75]:
bow_results_df["representation"] = "BoW"
tfidf_results_df["representation"] = "TF-IDF"

all_results = pd.concat(
    [bow_results_df, tfidf_results_df],
    ignore_index=True
)

all_results.sort_values(
    "f1",
    ascending=False
)

,configuration,vocabulary_size,accuracy,precision,recall,f1,representation
3,unigram_bigram_min_df2,482094,0.894379,0.890268,0.900552,0.895381,BoW
1,unigram_bigram,2322536,0.892362,0.884464,0.903566,0.893913,BoW
2,unigram_bigram_30000,30000,0.890345,0.890171,0.891512,0.890841,BoW
6,unigram_bigram_min_df2,482094,0.885304,0.872093,0.904068,0.887793,TF-IDF
4,unigram,82824,0.883287,0.871595,0.900050,0.885594,TF-IDF
7,unigram_bigram_sublinear,2322536,0.879506,0.867411,0.897037,0.881975,TF-IDF
5,unigram_bigram,2322536,0.878498,0.864669,0.898543,0.881281,TF-IDF
0,unigram,82824,0.875473,0.871096,0.882471,0.876747,BoW


### BoW vs TF-IDF — Validation Observations

- Among the configurations tested, **BoW achieved a higher validation F1 score than TF-IDF**.
- The best BoW configuration achieved an F1 score of approximately **0.8954**, while the best TF-IDF configuration achieved approximately **0.8878**.
- Therefore, BoW was the stronger representation for this sentiment-classification experiment based on the validation results.
- Both methods used the same training and validation splits and the same Logistic Regression classifier, making the comparison more consistent.

### Final BoW test

In [76]:
final_bow_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(
        ngram_range=(1, 2),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

final_bow_pipeline.fit(X_train_clean, y_train)



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizer', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [77]:
y_test_bow_pred = final_bow_pipeline.predict(X_test_clean)

In [78]:
print("BoW — Test Evaluation")
evaluate_model(y_test, y_test_bow_pred, "test")

BoW — Test Evaluation
test Accuracy : 0.9028
test Precision: 0.8990
test Recall   : 0.9084
test F1 Score : 0.9037


### Final TF-IDF test

In [79]:
final_tfidf_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

final_tfidf_pipeline.fit(X_train_clean, y_train)

y_test_tfidf_pred = final_tfidf_pipeline.predict(X_test_clean)

In [80]:
print("TF-IDF — Test Evaluation")
evaluate_model(y_test, y_test_tfidf_pred, "test")

TF-IDF — Test Evaluation
test Accuracy : 0.8959
test Precision: 0.8856
test Recall   : 0.9102
test F1 Score : 0.8977


### Create final test results dataframe

In [81]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

final_results = pd.DataFrame({
    "representation": [
        "BoW",
        "TF-IDF"
    ],
    
    "configuration": [
        "ngram_range=(1,2), min_df=2",
        "ngram_range=(1,2), min_df=2"
    ],
    
    "accuracy": [
        accuracy_score(y_test, y_test_bow_pred),
        accuracy_score(y_test, y_test_tfidf_pred)
    ],
    
    "precision": [
        precision_score(y_test, y_test_bow_pred),
        precision_score(y_test, y_test_tfidf_pred)
    ],
    
    "recall": [
        recall_score(y_test, y_test_bow_pred),
        recall_score(y_test, y_test_tfidf_pred)
    ],
    
    "f1": [
        f1_score(y_test, y_test_bow_pred),
        f1_score(y_test, y_test_tfidf_pred)
    ]
})

final_results

,representation,configuration,accuracy,precision,recall,f1
0,BoW,"ngram_range=(1,2), min_df=2",0.902793,0.898986,0.908379,0.903658
1,TF-IDF,"ngram_range=(1,2), min_df=2",0.895936,0.885630,0.910187,0.897741


In [82]:
final_results["classifier"] = "LogisticRegression"

In [85]:
final_results

,representation,configuration,accuracy,precision,recall,f1,classifier
0,BoW,"ngram_range=(1,2), min_df=2",0.902793,0.898986,0.908379,0.903658,LogisticRegression
1,TF-IDF,"ngram_range=(1,2), min_df=2",0.895936,0.885630,0.910187,0.897741,LogisticRegression


### Final Test Observations

- The best configurations selected using the validation set were evaluated on the previously untouched test set.
- The test set was used only for final evaluation and was not used during hyperparameter or configuration selection.
- The final test results provide the performance of the selected BoW and TF-IDF configurations on unseen data.
- The test results should be compared with the validation results to check whether the selected configurations generalize beyond the validation set.

### Saving the results 

In [86]:
final_results.to_csv(
    "D:/NLP-Projects/sentiment-analysis-imdb/results/bow_tfidf_results.csv",
    index=False
)

## Experimental Notes

### Dataset Split

The dataset was divided once in `01_eda_and_data_split.ipynb` using a **72% / 8% / 20%** train-validation-test split.

- **72% Training set:** used to fit the vectorizer and classifier.
- **8% Validation set:** used for hyperparameter/configuration selection.
- **20% Test set:** kept untouched until the final evaluation.

The same fixed train, validation, and test sets are used throughout the project so that the different text-representation techniques can be compared fairly.

### BoW and TF-IDF Training

Both BoW and TF-IDF were implemented using a scikit-learn `Pipeline` containing:

1. A text vectorizer (`CountVectorizer` or `TfidfVectorizer`)
2. A `LogisticRegression` classifier

The vectorizers were fitted only on the training data. Validation and test data were transformed using the already-fitted vectorizer.

### Hyperparameter/Configuration Selection

Several vectorizer configurations were evaluated on the validation set.

The validation set was used to select the best-performing configuration based primarily on **F1 score**.

The test set was not used during configuration selection.

### Final Test Evaluation

After selecting the best BoW and TF-IDF configurations using the validation set, each selected pipeline was evaluated on the test set.

The test set was therefore used only for the final performance evaluation.

### Training + Validation

The training and validation sets were **not combined** after model selection.

The final BoW and TF-IDF models used for test evaluation were trained on the original **72% training set** using the configurations selected through validation.

This was intentionally kept consistent with the evaluation approach used for the BERT model in this project.

### Important Data-Leakage Rule

The test set was never used to:

- fit the vectorizer
- train the classifier
- select hyperparameters
- select preprocessing configurations
- make modelling decisions

Therefore, the test results represent an evaluation on data that remained unseen during model development.